# 🌍 PyClimaExplorer — Dataset Exploration Notebook

Use this notebook to:
- Load and inspect a NetCDF climate dataset
- Explore its variables, dimensions, and attributes
- Generate quick matplotlib previews
- Validate data before loading it into the Streamlit dashboard

In [ ]:
import numpy as np
import pandas as pd
import xarray as xr
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import warnings
warnings.filterwarnings('ignore')

# Make figures look nicer
plt.rcParams.update({
    'figure.facecolor': '#0f1117',
    'axes.facecolor':   '#141824',
    'axes.edgecolor':   '#2a3a4a',
    'axes.labelcolor':  '#c9d1d9',
    'xtick.color':      '#c9d1d9',
    'ytick.color':      '#c9d1d9',
    'text.color':       '#c9d1d9',
    'grid.color':       '#2a3a4a',
    'grid.alpha':       0.5,
})

print('Libraries loaded ✓')

## 1. Load Dataset

Either load from a file or use the synthetic demo generator.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

from modules.data_loader import generate_synthetic_dataset

# Option A: Use the synthetic demo dataset
ds = generate_synthetic_dataset()

# Option B: Load your own NetCDF file
# ds = xr.open_dataset('../data/your_file.nc', use_cftime=True)

print(ds)

## 2. Dataset Overview

In [ ]:
print(f"Variables   : {list(ds.data_vars)}")
print(f"Dimensions  : {dict(ds.dims)}")
print(f"Coordinates : {list(ds.coords)}")
print()
print('Global attributes:')
for k, v in ds.attrs.items():
    print(f'  {k}: {v}')

## 3. Variable Detail

In [ ]:
variable = 'temperature'  # Change to any variable in the dataset

da = ds[variable]
print(da)
print(f"\nShape  : {da.shape}")
print(f"Min    : {float(da.min()):.3g}")
print(f"Max    : {float(da.max()):.3g}")
print(f"Mean   : {float(da.mean()):.3g}")
print(f"NaN %  : {float(da.isnull().mean())*100:.2f}%")

## 4. Spatial Preview — Single Time Step

In [ ]:
fig, ax = plt.subplots(figsize=(14, 6))

da.isel(time=0).plot(
    ax=ax,
    cmap='RdBu_r',
    robust=True,
    cbar_kwargs={'label': f"{variable} [{da.attrs.get('units', '')}", 'shrink': 0.7},
)

try:
    t0_label = str(pd.Timestamp(ds.time.values[0]))[:10]
except Exception:
    t0_label = 'step 0'

ax.set_title(f'{variable}  |  {t0_label}', fontsize=14, fontweight='bold')
ax.set_xlabel('Longitude')
ax.set_ylabel('Latitude')
ax.grid(True, linestyle='--', alpha=0.4)
plt.tight_layout()
plt.show()

## 5. Time Series at a Point Location

In [ ]:
from modules.utils import extract_point_timeseries

lat, lon = 23.5, 77.0       # Madhya Pradesh, India
ts = extract_point_timeseries(ds[variable], lat, lon)

fig, ax = plt.subplots(figsize=(12, 4))
ax.plot(ts.index, ts.values, color='#4a90d9', lw=1.5, label=variable)
ts.rolling(3, center=True, min_periods=1).mean().plot(
    ax=ax, color='#f5a623', lw=2.5, ls='--', label='3-step rolling mean'
)
ax.set_title(f'{variable} at ({lat}°N, {lon}°E)', fontsize=13, fontweight='bold')
ax.set_xlabel('Date')
ax.set_ylabel(f"{variable} [{da.attrs.get('units', '')}]")
ax.legend()
ax.grid(True, alpha=0.4)
plt.tight_layout()
plt.show()

## 6. Monthly Climatology & Anomaly

In [ ]:
from modules.utils import compute_monthly_anomaly

anomaly = compute_monthly_anomaly(ts)
colors  = ['#e74c3c' if v > 0 else '#3498db' for v in anomaly.values]

fig, axes = plt.subplots(2, 1, figsize=(12, 7), sharex=True)

# Raw
axes[0].plot(ts.index, ts.values, color='#4a90d9', lw=1.5)
axes[0].set_title(f'{variable} – raw values', fontweight='bold')
axes[0].set_ylabel(da.attrs.get('units', ''))
axes[0].grid(True, alpha=0.4)

# Anomaly bars
axes[1].bar(anomaly.index, anomaly.values, color=colors, width=20)
axes[1].axhline(0, color='#ccc', lw=0.8, ls='--')
axes[1].set_title(f'{variable} – monthly anomaly', fontweight='bold')
axes[1].set_ylabel(f'Δ {da.attrs.get("units", "")}')
axes[1].grid(True, alpha=0.4)

plt.tight_layout()
plt.show()

## 7. Export Spatial Slice as CSV

Useful for sharing or importing into GIS tools.

In [ ]:
da_2d = ds[variable].isel(time=0)
df_out = da_2d.to_dataframe(name=variable).reset_index()
df_out.head()

In [ ]:
out_path = f'../{variable}_spatial_t0.csv'
df_out.to_csv(out_path, index=False)
print(f'Saved to {out_path}')

---
**Next step:** Launch the full interactive dashboard with:
```bash
streamlit run ../app.py
```